In [ ]:
!pip install scikit-posthocs

## Metric-wise Model Performance Comparisons

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import kruskal, mannwhitneyu
from math import pi
from collections import defaultdict
import scikit_posthocs as sp
import itertools

# Load the data
file_path = '/content/ETL_Data_Question-wise.csv'
data = pd.read_csv(file_path)

# Quick view of the data
print(data.head())

# Ensure data types are correct
for col in ['clarity', 'clinical_relevance', 'option_accuracy', 'assessment_accuracy', 'feedback_quality']:
    data[col] = pd.to_numeric(data[col])

# Function to calculate mean ± SD
def calculate_mean_sd(metric, model):
    metric_data = data[data['model'] == model][metric]
    mean = metric_data.mean()
    sd = metric_data.std()
    return f"{mean:.2f}±{sd:.2f}"

# Function to calculate Cliff's Delta (non-parametric effect size)
def cliffs_delta(data1, data2):
    """Calculates Cliff's delta for two independent samples."""
    n1 = len(data1)
    n2 = len(data2)
    if n1 == 0 or n2 == 0:
        return 0.0, "No effect"  # Return 0 and "No effect" if either sample is empty
    numerator = 0
    for x in data1:
        for y in data2:
            if x > y:
                numerator += 1
            elif x < y:
                numerator -= 1
    delta = numerator / (n1 * n2)

    # Interpretation
    effect_size = "No effect"
    if abs(delta) >= 0.11 and abs(delta) < 0.33:
        effect_size = "Small effect"
    elif abs(delta) >= 0.33 and abs(delta) < 0.47:
        effect_size = "Medium effect"
    elif abs(delta) >= 0.47:
        effect_size = "Large effect"
    return delta, effect_size


# Statistical Tests and Results for Models
def model_level_stats_ranking(data):
    metrics = ['clarity', 'clinical_relevance', 'option_accuracy',
               'assessment_accuracy', 'feedback_quality']
    results_list = []

    for metric in metrics:
        model_groups = [data[data['model'] == model][metric] for model in data['model'].unique()]
        models = data['model'].unique()

        # Kruskal-Wallis Test
        kruskal_stat, kruskal_p = kruskal(*model_groups)
        kruskal_significant = kruskal_p < 0.05

        dunn_results = None
        if kruskal_significant and len(models) > 2: # Dunn's only if KW is significant and >2 groups
            try:
                dunn_results = sp.posthoc_dunn(model_groups, p_adjust='bonferroni')
            except Exception as e:
                print(f"Error during posthoc_dunn for {metric}: {e}")
                dunn_results = None

        # Pairwise Mann-Whitney U and Cliff's Delta
        for model1, model2 in itertools.combinations(models, 2):
            model1_data = data[data['model'] == model1][metric]
            model2_data = data[data['model'] == model2][metric]

            u_stat, u_p = mannwhitneyu(model1_data, model2_data, alternative='two-sided')
            delta, effect_size = cliffs_delta(model1_data, model2_data)

            model1_mean_sd = calculate_mean_sd(metric, model1)
            model2_mean_sd = calculate_mean_sd(metric, model2)

            # Determine Result String based on Mann-Whitney U and means
            if u_p < 0.05:
                if model1_data.mean() > model2_data.mean():
                    result_str = f"{model1} > {model2}"
                else:
                    result_str = f"{model1} < {model2}"
            else:
                result_str = f"{model1} = {model2}"

            dunn_p_value = None
            if dunn_results is not None and model1 in models and model2 in models:
                model1_index = list(models).index(model1)
                model2_index = list(models).index(model2)
                # Dunn's results is a matrix, accessing by index
                dunn_p_value = dunn_results.iloc[min(model1_index, model2_index), max(model1_index, model2_index)]


            results_list.append({
                'metric': metric,
                'model1': model1,
                'model2': model2,
                'u_stat': u_stat,
                'u_p': u_p,
                'cliff_delta': delta,
                'effect_size': effect_size,
                'model1_mean_sd': model1_mean_sd,
                'model2_mean_sd': model2_mean_sd,
                'result': result_str,
                'kruskal_stat': kruskal_stat,
                'kruskal_p': kruskal_p,
                'kruskal_significant': kruskal_significant,
                'dunn_p': dunn_p_value
            })

    results_df = pd.DataFrame(results_list)
    return results_df

updated_posthoc_results_df = model_level_stats_ranking(data)

# Save updated post-hoc results
updated_posthoc_results_df.to_csv('/content/Metric-wise_Model_Ranking_Table.csv', index=False)

print(updated_posthoc_results_df)

# Radar Chart for Model-Level Comparison (remains the same)
def plot_single_radar_chart(data):
    metrics = ['clarity', 'clinical_relevance', 'option_accuracy', 'assessment_accuracy', 'feedback_quality']
    models = data['model'].unique()
    num_metrics = len(metrics)

    # Custom radar scale: Uneven distances between intervals
    custom_scale = [0, 0.5, 1.25, 2.25, 3.5, 5] # Cumulative distances: 0-1 (0.5), 1-2 (0.75), 2-3 (1), 3-4 (1.25), 4-5 (1.5)
    custom_labels = ['0', '1', '2', '3', '4', '5'] # Labels for the scale

    # Prepare data for radar chart
    categories = list(metrics)
    angles = [n / float(num_metrics) * 2 * pi for n in range(num_metrics)]
    angles += angles[:1] # Complete the circle

    fig, ax = plt.subplots(figsize=(8, 8), subplot_kw={'polar': True})

    for model in models:
        model_data = data[data['model'] == model][metrics].mean()
        values = model_data.tolist()
        values += values[:1] # Complete the circle

        # Plot radar chart for the model
        ax.plot(angles, values, label=model)
        ax.fill(angles, values, alpha=0.1)

    # Set custom scale and labels
    ax.set_yticks(custom_scale)
    ax.set_yticklabels(custom_labels, fontsize=10)

    # Set category labels
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(categories, fontsize=10)

    # Add legend
    ax.legend(loc='upper right', bbox_to_anchor=(0.3, 1.2))

    plt.title("Model Comparison Across Metrics", size=15, y=-0.1)
    plt.tight_layout()
    plt.show()

plot_single_radar_chart(data)


def rank_models(metric, data): # Borda count ranking remains the same logic
    models = data['model'].unique()
    model_data = {model: data[data['model'] == model][metric] for model in models}

    # Create comparisons dictionary
    comparisons = {}
    for i, model1 in enumerate(models):
        for j, model2 in enumerate(models):
            if i < j: # Only process each pair once
                stat, p_value = mannwhitneyu(model_data[model1], model_data[model2])
                if p_value < 0.05:
                    if model_data[model1].mean() > model_data[model2].mean():
                        comparisons[(model1, model2)] = '>'
                    else:
                        comparisons[(model2, model1)] = '>'
                else:
                    comparisons[(model1, model2)] = '='

    # Initialize scoring system
    scores = defaultdict(int)

    # Process each pairwise comparison
    for (model1, model2), relation in comparisons.items():
        if relation == '>':
            scores[model1] += 1
            scores[model2] -= 1
        elif relation == '=':
            scores[model1] += 0
            scores[model2] += 0

    # Add any missing models with score 0
    for model in models:
        if model not in scores:
            scores[model] = 0

    # Group models by their scores
    score_groups = {}
    for model, score in scores.items():
        score_groups.setdefault(score, []).append(model)

    # Sort groups in descending order
    sorted_scores = sorted(score_groups.items(), key=lambda x: x[0], reverse=True)

    # Build ranking hierarchy
    ranking = []
    for score, models_list in sorted_scores:
        if len(models_list) > 1:
            ranking.append('='.join(sorted(models_list)))
        else:
            ranking.append(models_list[0])

    return ' > '.join(ranking)

# Calculate rankings for each metric
rankings = {metric: rank_models(metric, data) for metric in ['clarity', 'clinical_relevance', 'option_accuracy', 'assessment_accuracy', 'feedback_quality']}


# Print rankings for models
print("Rankings for Models:")
for metric, ranking in rankings.items():
    print(f"{metric}: {ranking}")

# Borda Count Calculation for Overall Ranking (remains the same)
rankings_str = rankings  # Use the rankings calculated above
models = list(data['model'].unique()) # Get models from data
num_models = len(models)
borda_scores = {model: 0 for model in models}

for metric, ranking_string in rankings_str.items():
    ranked_models = []
    rank_groups = ranking_string.split(' > ')
    current_rank = 1
    for group in rank_groups:
        tied_models = group.split('=')
        num_tied = len(tied_models)

        # Calculate points for these ranks
        points_for_ranks = []
        for r in range(current_rank, current_rank + num_tied):
            points_for_ranks.append(max(0, num_models - r + 1)) # Points cannot be negative

        avg_points = sum(points_for_ranks) / num_tied if num_tied > 0 else 0 # Average points for tied models

        for model in tied_models:
            borda_scores[model] += avg_points

        current_rank += num_tied # Increment rank for the next group

# Sort models by Borda scores in descending order
ranked_models_overall = sorted(borda_scores.items(), key=lambda item: item[1], reverse=True)

print("\nOverall Rankings (Borda Count):")
overall_ranking_string_list = []
for i, (model, score) in enumerate(ranked_models_overall):
    print(f"{i+1}. {model}: {score:.1f} points")
    overall_ranking_string_list.append(model)

overall_ranking_string = ' > '.join(overall_ranking_string_list)
print(f"\nOverall Ranking String (Borda Count): {overall_ranking_string}")

## System-wise Model Performance Comparisons

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import kruskal, mannwhitneyu
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from math import pi
from collections import defaultdict
import scikit_posthocs as sp
import itertools

# Load the data
file_path = '/content/ETL_Data_Question-wise.csv'
data = pd.read_csv(file_path)

# Quick view of the data
print(data.head())

# Function to calculate Cliff's Delta (non-parametric effect size)
def cliffs_delta(data1, data2):
    """Calculates Cliff's delta for two independent samples."""
    n1 = len(data1)
    n2 = len(data2)
    if n1 == 0 or n2 == 0:
        return 0.0, "No effect"  # Return 0 and "No effect" if either sample is empty
    numerator = 0
    for x in data1:
        for y in data2:
            if x > y:
                numerator += 1
            elif x < y:
                numerator -= 1
    delta = numerator / (n1 * n2)

    # Interpretation
    effect_size = "No effect"
    if abs(delta) >= 0.11 and abs(delta) < 0.33:
        effect_size = "Small effect"
    elif abs(delta) >= 0.33 and abs(delta) < 0.47:
        effect_size = "Medium effect"
    elif abs(delta) >= 0.47:
        effect_size = "Large effect"
    return delta, effect_size

# Function to calculate mean ± SD
def calculate_mean_sd(metric, model, system_data):
    metric_data = system_data[system_data['model'] == model][metric]
    mean = metric_data.mean()
    sd = metric_data.std()
    return f"{mean:.2f}±{sd:.2f}"

# Ensure data types are correct
for col in ['clarity', 'clinical_relevance', 'option_accuracy', 'assessment_accuracy', 'feedback_quality']:
    data[col] = pd.to_numeric(data[col])

# System-Level Comparison with Non-Parametric Tests
def system_level_posthoc_ranking(data):
    metrics = ['clarity', 'clinical_relevance', 'option_accuracy',
               'assessment_accuracy', 'feedback_quality']
    comprehensive_results = []
    systems = data['system'].unique()

    for system in systems:
        system_data = data[data['system'] == system]
        models_in_system = system_data['model'].unique()

        for metric in metrics:
            model_groups = [system_data[system_data['model'] == model][metric] for model in models_in_system]
            models_and_groups = list(zip(models_in_system, model_groups))

            # Identify models with constant values
            constant_models = [model for model, group in models_and_groups if len(group.unique()) <= 1]
            has_constant = len(constant_models) > 0

            # Initialize Kruskal-Wallis results
            kruskal_stat = np.nan
            kruskal_p = np.nan
            kruskal_significant = False
            kruskal_skip_reason = ""
            dunn_results = None

            # Handle Kruskal-Wallis execution or skipping
            if has_constant:
                kruskal_skip_reason = (
                    f"Kruskal-Wallis skipped due to constant values in model(s): {', '.join(constant_models)}"
                    if constant_models else ""
                )
            else:
                try:
                    if len(models_in_system) > 1:
                        kruskal_stat, kruskal_p = kruskal(*model_groups)
                        kruskal_significant = kruskal_p < 0.05

                        if kruskal_significant and len(models_in_system) > 2:
                            dunn_results = sp.posthoc_dunn(model_groups, p_adjust='bonferroni')
                except Exception as e:
                    kruskal_skip_reason = f"Error during Kruskal-Wallis: {str(e)}"
                    kruskal_stat = np.nan
                    kruskal_p = np.nan

            # Process all pairwise comparisons
            for model1, model2 in itertools.combinations(models_in_system, 2):
                # Get data for current pair
                model1_data = system_data[system_data['model'] == model1][metric]
                model2_data = system_data[system_data['model'] == model2][metric]

                # Mann-Whitney U test
                u_stat, u_p = mannwhitneyu(model1_data, model2_data, alternative='two-sided')

                # Cliff's Delta effect size
                delta, effect_size = cliffs_delta(model1_data, model2_data)

                # Format mean ± SD
                model1_mean_sd = calculate_mean_sd(metric, model1, system_data)
                model2_mean_sd = calculate_mean_sd(metric, model2, system_data)

                # Determine comparison result
                result_str = f"{model1} = {model2}"
                if u_p < 0.05:
                    if model1_data.mean() > model2_data.mean():
                        result_str = f"{model1} > {model2}"
                    else:
                        result_str = f"{model1} < {model2}"

                # Get Dunn's p-value if available
                dunn_p_value = None
                if dunn_results is not None:
                    model1_idx = list(models_in_system).index(model1)
                    model2_idx = list(models_in_system).index(model2)
                    dunn_p_value = dunn_results.iloc[model1_idx, model2_idx]

                # Append results with skip reason
                comprehensive_results.append({
                    'system': system,
                    'metric': metric,
                    'model1': model1,
                    'model2': model2,
                    'u_stat': u_stat,
                    'u_p': u_p,
                    'cliff_delta': delta,
                    'effect_size': effect_size,
                    'model1_mean_sd': model1_mean_sd,
                    'model2_mean_sd': model2_mean_sd,
                    'result': result_str,
                    'kruskal_stat': kruskal_stat,
                    'kruskal_p': kruskal_p,
                    'kruskal_significant': kruskal_significant,
                    'dunn_p': dunn_p_value,
                    'kruskal_skip_reason': kruskal_skip_reason
                })

    # Create DataFrame with explicit column order
    columns = [
        'system', 'metric', 'model1', 'model2', 'u_stat', 'u_p', 'cliff_delta',
        'effect_size', 'model1_mean_sd', 'model2_mean_sd', 'result',
        'kruskal_stat', 'kruskal_p', 'kruskal_significant', 'dunn_p',
        'kruskal_skip_reason'
    ]

    return pd.DataFrame(comprehensive_results, columns=columns)

system_posthoc_results_df = system_level_posthoc_ranking(data)

# Save the comprehensive system-level post-hoc results to a single CSV
system_posthoc_results_df.to_csv('/content/System-wise_Model_Ranking_Table.csv', index=False)

print(system_posthoc_results_df)


def rank_models_system_level(metric, system_data):
    models = system_data['model'].unique()
    model_data = {model: system_data[system_data['model'] == model][metric] for model in models}

    # Create comparisons dictionary
    comparisons = {}
    for i, model1 in enumerate(models):
        for j, model2 in enumerate(models):
            if i < j: # Only process each pair once
                stat, p_value = mannwhitneyu(model_data[model1], model_data[model2])
                if p_value < 0.05:
                    if model_data[model1].mean() > model_data[model2].mean():
                        comparisons[(model1, model2)] = '>'
                    else:
                        comparisons[(model2, model1)] = '>'
                else:
                    comparisons[(model1, model2)] = '='

    # Initialize scoring system
    scores = defaultdict(int)

    # Process each pairwise comparison
    for (model1, model2), relation in comparisons.items():
        if relation == '>':
            scores[model1] += 1
            scores[model2] -= 1
        elif relation == '=':
            scores[model1] += 0
            scores[model2] += 0

    # Add any missing models with score 0
    for model in models:
        if model not in scores:
            scores[model] = 0

    # Group models by their scores
    score_groups = {}
    for model, score in scores.items():
        score_groups.setdefault(score, []).append(model)

    # Sort groups in descending order
    sorted_scores = sorted(score_groups.items(), key=lambda x: x[0], reverse=True)

    # Build ranking hierarchy
    ranking = []
    for score, models_list in sorted_scores:
        if len(models_list) > 1:
            ranking.append('='.join(sorted(models_list)))
        else:
            ranking.append(models_list[0])

    return ' > '.join(ranking)


# Calculate rankings for each metric in each system
system_rankings = {}
for system in data['system'].unique():
    system_data = data[data['system'] == system]
    rankings = {metric: rank_models_system_level(metric, system_data) for metric in ['clarity', 'clinical_relevance', 'option_accuracy', 'assessment_accuracy', 'feedback_quality']}
    system_rankings[system] = rankings

# Print rankings for models in each system and metric
print("\nRankings for Models per System and Metric:")
for system, rankings in system_rankings.items():
    print(f"\nSystem: {system}")
    for metric, ranking in rankings.items():
        print(f"  {metric}: {ranking}")


# Borda Count Calculation for Overall Ranking for each system
system_borda_overall_rankings = {}
for system in data['system'].unique():
    system_data = data[data['system'] == system]
    rankings_str = system_rankings[system] # Use metric rankings calculated above for this system
    models = list(system_data['model'].unique())
    num_models = len(models)
    borda_scores = {model: 0 for model in models}

    for metric, ranking_string in rankings_str.items():
        ranked_models = []
        rank_groups = ranking_string.split(' > ')
        current_rank = 1
        for group in rank_groups:
            tied_models = group.split('=')
            num_tied = len(tied_models)

            # Calculate points for these ranks
            points_for_ranks = []
            for r in range(current_rank, current_rank + num_tied):
                points_for_ranks.append(max(0, num_models - r + 1)) # Points cannot be negative

            avg_points = sum(points_for_ranks) / num_tied if num_tied > 0 else 0 # Average points for tied models

            for model in tied_models:
                borda_scores[model] += avg_points

            current_rank += num_tied # Increment rank for the next group

    # Sort models by Borda scores in descending order
    ranked_models_overall = sorted(borda_scores.items(), key=lambda item: item[1], reverse=True)

    overall_ranking_string_list = [model for model, score in ranked_models_overall]
    overall_ranking_string = ' > '.join(overall_ranking_string_list)
    system_borda_overall_rankings[system] = overall_ranking_string


# Print Final Borda Count Overall Ranking for each system
print("\nFinal Borda Count Overall Rankings per System:")
for system, ranking in system_borda_overall_rankings.items():
    print(f"System: {system}: {ranking}")


## Polar Chart for System-wise Model Comparisons

In [ ]:
# Radar Chart Plotting Function
def plot_radar_charts(data):
    metrics = ['clarity', 'clinical_relevance', 'option_accuracy', 'assessment_accuracy', 'feedback_quality']
    systems = data['system'].unique()
    models_unique = data['model'].unique() # Get unique models for legend
    num_metrics = len(metrics)

    num_cols = 3  # Number of charts per row
    num_rows = -(-len(systems) // num_cols)  # Ceiling division for rows

    fig, axes = plt.subplots(num_rows, num_cols, figsize=(15, 5 * num_rows), subplot_kw={'polar': True})
    axes = axes.flatten()  # Flatten to easily iterate even if single row

    # Custom radar scale: Uneven distances between intervals (defined once here)
    custom_scale = [0, 0.5, 1.25, 2.25, 3.5, 5]
    custom_labels = ['', '1', '2', '3', '4', '5']

    # Create a dictionary to hold lines for legend
    legend_lines = {}

    for ax, system in zip(axes, systems):
        system_data = data[data['system'] == system].groupby('model')[metrics].mean()

        # Prepare data for radar chart
        categories = list(metrics)
        angles = [n / float(num_metrics) * 2 * pi for n in range(num_metrics)]
        angles += angles[:1]

        for model in system_data.index:
            values = system_data.loc[model].tolist()
            values += values[:1]  # Complete the circle

            # Plot each model, store the line object for legend
            line, = ax.plot(angles, values, label=model) # Get line object
            ax.fill(angles, values, alpha=0.1)
            legend_lines[model] = line # Store line object for legend

        ax.set_title(system, size=15, y=1.1)
        ax.set_xticks(angles[:-1])
        ax.set_xticklabels(categories, fontsize=10)

        ax.set_yticks(custom_scale)
        ax.set_yticklabels(custom_labels, fontsize=10)

    # Hide any unused subplots
    for ax in axes[len(systems):]:
        ax.axis('off')

    # Create legend handles and labels from legend_lines
    handles = [legend_lines[model] for model in models_unique]
    labels = list(models_unique) # Ensure labels are in consistent order

    fig.legend(handles, labels, loc='upper right', bbox_to_anchor=(1.05, 0.95))
    plt.tight_layout(rect=[0, 0, 0.9, 1.0])
    plt.show()

plot_radar_charts(data)



In [ ]:
# Borda Count Calculation for Final Overall Ranking Across Systems

# Calculate rankings for each metric in each system
system_rankings = {}
for system in data['system'].unique():
    system_data = data[data['system'] == system]
    rankings = {metric: rank_models_system_level(metric, system_data) for metric in ['clarity', 'clinical_relevance', 'option_accuracy', 'assessment_accuracy', 'feedback_quality']}
    system_rankings[system] = rankings

# Borda Count Calculation for Final Overall Ranking ACROSS ALL Systems
overall_borda_scores = defaultdict(float) # Use defaultdict(float) to initialize scores to 0.0

for system in data['system'].unique():
    rankings_str = system_rankings[system]
    models_in_system = list(data[data['system'] == system]['model'].unique()) # Models in current system
    num_models_in_system = len(models_in_system)

    for metric, ranking_string in rankings_str.items():
        ranked_models = []
        rank_groups = ranking_string.split(' > ')
        current_rank = 1
        for group in rank_groups:
            tied_models = group.split('=')
            num_tied = len(tied_models)

            # Calculate points for these ranks (using num_models_in_system for points)
            points_for_ranks = []
            for r in range(current_rank, current_rank + num_tied):
                points_for_ranks.append(max(0, num_models_in_system - r + 1))

            avg_points = sum(points_for_ranks) / num_tied if num_tied > 0 else 0

            for model in tied_models:
                overall_borda_scores[model] += avg_points # Accumulate scores ACROSS systems

            current_rank += num_tied

# Sort models by accumulated Borda scores in descending order
ranked_models_overall_final = sorted(overall_borda_scores.items(), key=lambda item: item[1], reverse=True)

print("\nFinal OVERALL Borda Count Ranking (Aggregated Across Systems):")
overall_ranking_string_list_final = []
for i, (model, score) in enumerate(ranked_models_overall_final):
    print(f"{i+1}. {model}: {score:.1f} points")
    overall_ranking_string_list_final.append(model)

overall_ranking_string_final = ' > '.join(overall_ranking_string_list_final)
print(f"\nFinal Overall Ranking String (Across Systems): {overall_ranking_string_final}")


Final OVERALL Borda Count Ranking (Aggregated Across Systems):
1. gpt-4o: 146.5 points
2. llama3-70b-8192: 143.5 points
3. anthropic/claude-3.5-sonnet: 121.0 points
4. gemini-1.5-flash-latest: 89.0 points

Final Overall Ranking String (Across Systems): gpt-4o > llama3-70b-8192 > anthropic/claude-3.5-sonnet > gemini-1.5-flash-latest


## Accuracy Metrics



In [ ]:
### ACCURACY METRICS


import pandas as pd
import numpy as np

# Load the data
file_path = '/content/ETL_Data_Question-wise.csv'
data = pd.read_csv(file_path)

# Ensure data types are correct for rating columns
metrics_cols = ['clarity', 'clinical_relevance', 'difficulty', 'option_accuracy', 'assessment_accuracy', 'feedback_quality']
for col in metrics_cols:
    data[col] = pd.to_numeric(data[col])

models = data['model'].unique()
raters_count = data['email'].nunique()

accuracy_definitions_selected = {
    'top_box': lambda scores: sum(scores == 5) >= 2,  # Top Box (at least 2 out of 3 raters = 5)
    'avg_score_4': lambda scores: np.mean(scores) >= 4, # Avg Score >= 4
    'top_two_box': lambda scores: all(scores >= 4),   # Top Two Boxes (all raters >= 4)
    'perfect_5': lambda scores: all(scores == 5)       # Perfect 5 (Original)
}

composite_accuracy_metrics = {
    'question_generation_accuracy': {
        'metrics': ['clarity', 'clinical_relevance', 'option_accuracy']
    },
    'overall_assessment_accuracy': {
        'metrics': ['assessment_accuracy', 'feedback_quality']
    }
}


accuracy_results_final = []

for model in models:
    model_data = data[data['model'] == model]
    total_questions_model = model_data['ques_id'].nunique()

    if total_questions_model == 0:
        accuracy_entry = {'model': model}
        for metric in metrics_cols:
            for def_name in accuracy_definitions_selected:
                accuracy_entry[f'{metric}_{def_name}_accuracy'] = 0
        for comp_metric_name in composite_accuracy_metrics:
            for def_name in accuracy_definitions_selected:
                accuracy_entry[f'{comp_metric_name}_{def_name}_accuracy'] = 0
        accuracy_results_final.append(accuracy_entry)
        continue

    metric_def_counts = {}
    for metric in metrics_cols:
        metric_def_counts[metric] = {def_name: 0 for def_name in accuracy_definitions_selected}

    composite_metric_def_counts = {}
    for comp_metric_name in composite_accuracy_metrics:
        composite_metric_def_counts[comp_metric_name] = {def_name: 0 for def_name in accuracy_definitions_selected}


    unique_question_ids = model_data['ques_id'].unique()

    for ques_id in unique_question_ids:
        question_data = model_data[model_data['ques_id'] == ques_id]

        for metric in metrics_cols:
            question_metric_scores = question_data[metric]
            for def_name, definition_func in accuracy_definitions_selected.items():
                if definition_func(question_metric_scores):
                    metric_def_counts[metric][def_name] += 1

        # Calculate Composite Accuracy Metrics for all definitions
        for comp_metric_name, comp_config in composite_accuracy_metrics.items():
            comp_metrics_list = comp_config['metrics']
            for def_name, definition_func in accuracy_definitions_selected.items():
                if all(definition_func(question_data[metric]) for metric in comp_metrics_list):
                    composite_metric_def_counts[comp_metric_name][def_name] += 1


    accuracy_entry = {'model': model}
    for metric in metrics_cols:
        for def_name in accuracy_definitions_selected:
            accuracy_percentage = (metric_def_counts[metric][def_name] / total_questions_model) * 100 if total_questions_model > 0 else 0
            accuracy_entry[f'{metric}_{def_name}_accuracy'] = accuracy_percentage

    for comp_metric_name in composite_accuracy_metrics:
        for def_name in accuracy_definitions_selected:
            accuracy_percentage = (composite_metric_def_counts[comp_metric_name][def_name] / total_questions_model) * 100 if total_questions_model > 0 else 0
            accuracy_entry[f'{comp_metric_name}_{def_name}_accuracy'] = accuracy_percentage

    accuracy_results_final.append(accuracy_entry)


accuracy_df_final = pd.DataFrame(accuracy_results_final)
print(accuracy_df_final)

# Save to CSV
accuracy_df_final.to_csv('/content/Accuracy_Metrics_Basic_Composite.csv', index=False)